In [0]:
employee=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/employee_project/employee_files/employees_data_quality_test.csv")

## Finding Invalid id

In [0]:
from pyspark.sql.functions import col,lit
employee_id_pattern=r"^EMP[0-9]{6}$"
invalid_employee=employee.filter(~col("employee_id").rlike(employee_id_pattern))
invalid_employee.show(truncate=False)

+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+-------------------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|city     |employee_status|email                    |
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+-------------------------+
|EMP-100005 |Nachiket  |Bath     |HR        |Manager    |EMP100001 |-55000|2025-02-14  |Hyderabad|Inactive       |nachiket.bath@company.com|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+-------------------------+



In [0]:
invalid_table=invalid_employee.withColumn("reject_reason",lit("Invalid Employee id"))
invalid_table.show()

+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+--------------------+-------------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|     city|employee_status|               email|      reject_reason|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+--------------------+-------------------+
| EMP-100005|  Nachiket|     Bath|        HR|    Manager| EMP100001|-55000|  2025-02-14|Hyderabad|       Inactive|nachiket.bath@com...|Invalid Employee id|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+--------------------+-------------------+



In [0]:
invalid_table.write.mode("overwrite").format("delta").saveAsTable("employee_project.invalid_id")

In [0]:
%sql
select * from employee_project.invalid_id

employee_id,first_name,last_name,department,designation,manager_id,salary,joining_date,city,employee_status,email,reject_reason
EMP-100005,Nachiket,Bath,HR,Manager,EMP100001,-55000,2025-02-14,Hyderabad,Inactive,nachiket.bath@company.com,Invalid Employee id


In [0]:
email_pattern=r"^[A-Za-z.]+@[A-Za-z]+\.[A-Za-z]{3}$"
invalid_email=employee.filter(~col("email").rlike(email_pattern))
invalid_email.show(truncate=False)

+-----------+----------+---------+----------+-----------+----------+------+------------+------+---------------+-----------------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|city  |employee_status|email                  |
+-----------+----------+---------+----------+-----------+----------+------+------------+------+---------------+-----------------------+
|EMP100010  |Wriddhish |Dugar    |Sales     |Engineer   |EMP100001 |595442|2026-04-23  |Mumbai|Inactive       |wriddhish.dugar@company|
+-----------+----------+---------+----------+-----------+----------+------+------------+------+---------------+-----------------------+



In [0]:
invalid_email_table=invalid_email.withColumn("reject_reason",lit("Invalid Employee email"))
invalid_email_table.show()

+-----------+----------+---------+----------+-----------+----------+------+------------+------+---------------+--------------------+--------------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|  city|employee_status|               email|       reject_reason|
+-----------+----------+---------+----------+-----------+----------+------+------------+------+---------------+--------------------+--------------------+
|  EMP100010| Wriddhish|    Dugar|     Sales|   Engineer| EMP100001|595442|  2026-04-23|Mumbai|       Inactive|wriddhish.dugar@c...|Invalid Employee ...|
+-----------+----------+---------+----------+-----------+----------+------+------------+------+---------------+--------------------+--------------------+



In [0]:
invalid_email_table.write.mode("overwrite").format("delta").saveAsTable("employee_project.invalid_email")

In [0]:
%sql
select * from employee_project.invalid_email;

employee_id,first_name,last_name,department,designation,manager_id,salary,joining_date,city,employee_status,email,reject_reason
EMP100010,Wriddhish,Dugar,Sales,Engineer,EMP100001,595442,2026-04-23,Mumbai,Inactive,wriddhish.dugar@company,Invalid Employee email
